# 04 - Phase 4a: tonic/pitch extraction from raw audio, and B1/M1's sensitivity to tonic error

CPU is enough for Part A (essentia is DSP, not deep learning) and for Part B (B1). Part C (M1) is fast CPU
inference too -- no GPU strictly required anywhere in this notebook, unlike notebook 03's training.

**What this answers, and why it's built this way** (see PLAN.md Phase 4a and CONCEPTS.md for the full
reasoning -- this is a summary, not a repeat):
- B1/M1 were trained and evaluated on *pre-extracted* pitch and *dataset-provided* tonic. This notebook
  builds and validates what the real app needs instead: pitch/tonic estimated from raw audio.
- Re-running B1/M1 with a *real* estimated tonic isn't possible on the existing split -- 298/306
  recordings are HMD, whose audio we don't have. So this is two independent pieces instead:
  **Part A** validates the extractor against Saraga's *full* audio catalog (doesn't need raga labels,
  so Saraga's thinness within our 30-raga list doesn't matter here). **Part B/C** simulate tonic error
  by injecting offsets into the *already-known* oracle tonic across the full 306-recording split, and
  evaluate the already-frozen B1/M1 at inference time only (no retraining -- matches real deployment).
  **Part D** combines them: reads Part A's real-world error magnitude off Part B/C's sensitivity curve.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/vishwesh-pujari/raaga-detector.git"
REPO_DIR = "/content/raaga-detector" if os.path.exists("/content") else "/kaggle/working/raaga-detector"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "essentia"], check=True)  # not preinstalled, unlike torch
sys.path.insert(0, f"{REPO_DIR}/src")  # editable installs are not always visible to a running kernel

# Google Drive is mounted by default: earlier notebooks' results (the split file, caches) are
# already there, so this picks them up automatically instead of asking you to re-upload/rebuild
# them. Set False to work on this session's local disk only -- the next cell then falls back to
# asking you to upload splits/v1.csv from your computer if it can't find it.
MOUNT_DRIVE = True  # set False to skip Drive and work fully locally this session
if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount failed or was skipped, continuing on local disk:", e)

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
import essentia
print("essentia", essentia.__version__)

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, json
import matplotlib.pyplot as plt
import torch
from raaga import env
from raaga.data import catalog, hmd, saraga
from raaga.features import extract, pitch as P
from raaga.features import cache as C, sequence_cache as SC
from raaga.eval import metrics, tonic_robustness as TR
from raaga.models import baseline, sequence as M

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RAW, ROOT = env.raw_data_home(), env.root()
split_path = ROOT / "splits" / "v1.csv"
if not split_path.exists():
    from google.colab import files
    import shutil
    print("splits/v1.csv not found under", ROOT, "-- choose the v1.csv file from notebook 01")
    uploaded = files.upload()
    assert len(uploaded) == 1
    split_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.move(next(iter(uploaded)), split_path)
split = pd.read_csv(split_path)
CLASSES = sorted(split.raga.unique())
N_CLASSES = len(CLASSES)
print(len(split), "recordings,", N_CLASSES, "ragas | device:", DEVICE)

## Part A: extractor validation against Saraga's full catalog
Downloads Saraga's audio this time (`include_audio=True` -- a much bigger download than notebooks
01-03 ever needed, since they deliberately skipped it). Uses a *separate* completion marker from the
earlier text-only download, so this correctly fetches the audio even if Saraga was already
downloaded without it in an earlier session (see `remote.py`/`saraga.py` -- this exact bug was
caught and fixed while building this notebook, not discovered by accident later).

In [ ]:
saraga.download(RAW, include_audio=True)
sar_df = pd.DataFrame(saraga.rows(RAW))
sar_df = sar_df[sar_df.tonic_hz.notna() & sar_df.audio_path.map(lambda p: Path(p).exists())]
print(len(sar_df), "Saraga recordings with both ground-truth tonic and audio available")

In [ ]:
def evaluate_one(row, sample_rate=44100):
    """Compare our extractor's output to Saraga's own shipped ground truth for one recording."""
    audio = extract.load_audio(row.audio_path, sample_rate)
    out = {"uid": row.uid, "true_tonic_hz": row.tonic_hz}
    try:
        out["est_tonic_hz"] = extract.estimate_tonic(audio, sample_rate)
        out["tonic_error_cents"] = 1200 * np.log2(out["est_tonic_hz"] / row.tonic_hz)
    except Exception as e:
        out["tonic_error_cents"] = None
        out["tonic_error"] = str(e)
    try:
        times, freqs = extract.extract_pitch(audio, sample_rate)
        true_t, true_f = P.load_pitch_file(row.pitch_path)
        # approximate align (nearest earlier frame) onto the ground truth's time grid -- our
        # extractor's frames (~2.9ms) are finer than the ground truth's (~4.4ms), so this is a
        # small, non-systematic approximation, fine for a median-error summary metric
        idx = np.searchsorted(times, true_t).clip(0, len(times) - 1)
        both_voiced = (freqs[idx] > 0) & (true_f > 0)
        if both_voiced.sum() > 100:
            cents_diff = 1200 * np.log2(freqs[idx][both_voiced] / true_f[both_voiced])
            out["pitch_median_abs_cents_err"] = float(np.median(np.abs(cents_diff)))
            out["pitch_within_50c_frac"] = float((np.abs(cents_diff) < 50).mean())
        out["voiced_frac_ours"] = float((freqs > 0).mean())
    except Exception as e:
        out["pitch_error"] = str(e)
    return out

# Checkpointed: ~5 hours estimated for the full catalog (see PLAN.md Phase 4a), so this writes
# a row to disk after every recording and skips already-done ones on a re-run -- a disconnect
# partway through loses only the in-flight recording, not everything before it.
(ROOT / "results").mkdir(exist_ok=True)
csv_path = ROOT / "results" / "extractor_validation.csv"
done = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()
done_uids = set(done.uid) if len(done) else set()
print(f"{len(done_uids)}/{len(sar_df)} already done" + (" -- resuming" if done_uids else ""))

results = done.to_dict("records")
for i, row in enumerate(sar_df.itertuples(index=False), 1):
    if row.uid in done_uids:
        continue
    results.append(evaluate_one(row))
    pd.DataFrame(results).to_csv(csv_path, index=False)  # checkpoint after every recording
    print(f"  {i}/{len(sar_df)}  {row.uid}")
extractor_val = pd.DataFrame(results)
display(extractor_val.describe())

In [ ]:
ok = extractor_val.dropna(subset=["tonic_error_cents"])
print(f"tonic estimated for {len(ok)}/{len(extractor_val)} recordings")
print("median |tonic error| (cents):", round(ok.tonic_error_cents.abs().median(), 1))
print("tonic within 50 cents:", round((ok.tonic_error_cents.abs() < 50).mean(), 3))
print("tonic within 100 cents (1 semitone):", round((ok.tonic_error_cents.abs() < 100).mean(), 3))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].hist(ok.tonic_error_cents.clip(-600, 600), bins=40)
ax[0].set_xlabel("tonic error (cents)"); ax[0].set_title("Part A: tonic error distribution")
if "pitch_median_abs_cents_err" in extractor_val:
    ax[1].hist(extractor_val.pitch_median_abs_cents_err.dropna(), bins=40)
    ax[1].set_xlabel("median |pitch error| per recording (cents)"); ax[1].set_title("pitch-track agreement")
plt.tight_layout(); plt.show()
REAL_TONIC_ERROR_CENTS = float(ok.tonic_error_cents.abs().median())
print("\nREAL_TONIC_ERROR_CENTS (used in Part D):", round(REAL_TONIC_ERROR_CENTS, 1))

## Part B: B1's sensitivity to tonic error
No audio needed -- offsets are injected into the already-known oracle tonic. Retrains B1 quickly
(seconds) on folds 0-3 with the *unperturbed* tonic, matching the frozen B1 config, then evaluates
that one fixed model against the held-out fold 4 recomputed at each offset.

In [ ]:
TONIC = "tonic"
hmd.download(RAW)
paths = catalog.build_catalog(RAW, include_saraga=True, tonic_kind=TONIC)[["uid", "pitch_path", "tonic_hz"]]
full_catalog = split.merge(paths, on="uid")
train_catalog = full_catalog[full_catalog.fold != 4]
test_catalog = full_catalog[full_catalog.fold == 4]

hist_params = dict(n_bins=120, chunk_s=30.0, hop_s=15.0, smooth_bins=1.0)
train_cache = C.cache_dir(ROOT, variant="phase4a_train", **hist_params)
C.build(train_catalog, train_cache, verbose=False, **hist_params)
train_data = C.load(train_catalog, train_cache)
b1_model = baseline.make_model(C=0.1).fit(train_data.X, train_data.y)  # C=0.1: B1's frozen choice
b1_classes = list(b1_model.classes_)
print("B1 retrained on", len(train_data.X), "chunks for the sensitivity sweep")

def b1_build_and_evaluate(perturbed_test_catalog):
    tag = f"{perturbed_test_catalog.tonic_hz.sum():.3f}"
    cache = C.cache_dir(ROOT, variant=f"phase4a_eval_{tag}", **hist_params)
    C.build(perturbed_test_catalog, cache, verbose=False, **hist_params)
    data = C.load(perturbed_test_catalog, cache)
    y_idx = np.array([b1_classes.index(y) for y in data.y])
    proba = b1_model.predict_proba(data.X)
    out = metrics.evaluate_predictions(proba, y_idx, data.uid, b1_classes)
    return {k: v for k, v in out.items() if k not in ("confusion", "classes")}

OFFSETS = [5, 10, 20, 35, 50, 75, 100, 150, 250, 400, 600]
b1_sweep = TR.sweep(test_catalog, OFFSETS, b1_build_and_evaluate)
display(b1_sweep[["offset_cents", "rec_top1", "rec_macro_f1"]])

## Part C: M1's sensitivity to tonic error
Loads the already-trained, already-saved `m1_weights.pt` -- inference only, no retraining.

In [ ]:
seq_params = dict(chunk_s=30.0, hop_s=15.0, step_s=0.1)
m1_model = M.PitchSequenceModel(M.ModelConfig(n_classes=N_CLASSES, conv_channels=(32, 64), gru_hidden=128))
m1_model.load_state_dict(torch.load(ROOT / "results" / "m1_weights.pt", map_location=DEVICE))
m1_model.to(DEVICE).eval()
print("loaded m1_weights.pt")

def m1_build_and_evaluate(perturbed_test_catalog):
    tag = f"{perturbed_test_catalog.tonic_hz.sum():.3f}"
    cache = SC.cache_dir(ROOT, variant=f"phase4a_eval_{tag}", **seq_params)
    SC.build(perturbed_test_catalog, cache, verbose=False, **seq_params)
    data = SC.load(perturbed_test_catalog, cache)
    y_idx = np.array([CLASSES.index(y) for y in data.y])
    proba = M.predict_proba(m1_model, data.X, device=DEVICE)
    out = metrics.evaluate_predictions(proba, y_idx, data.uid, CLASSES)
    return {k: v for k, v in out.items() if k not in ("confusion", "classes")}

m1_sweep = TR.sweep(test_catalog, OFFSETS, m1_build_and_evaluate)
display(m1_sweep[["offset_cents", "rec_top1", "rec_macro_f1"]])

## Part D: put it together

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(b1_sweep.offset_cents, b1_sweep.rec_top1, "o-", label="B1")
ax.plot(m1_sweep.offset_cents, m1_sweep.rec_top1, "o-", label="M1")
ax.axvline(REAL_TONIC_ERROR_CENTS, color="red", linestyle="--", label=f"measured real extractor error ({REAL_TONIC_ERROR_CENTS:.0f}c)")
ax.axhline(1 / N_CLASSES, color="gray", linestyle=":", label="chance")
ax.set_xlabel("injected tonic error (cents)"); ax.set_ylabel("rec_top1"); ax.legend(); ax.set_xscale("symlog")
plt.tight_layout(); plt.show()

def interp(sweep_df, x):
    return float(np.interp(x, sweep_df.offset_cents, sweep_df.rec_top1))

print(f"At the real extractor's measured tonic error ({REAL_TONIC_ERROR_CENTS:.0f} cents):")
print(f"  B1 estimated rec_top1: {interp(b1_sweep, REAL_TONIC_ERROR_CENTS):.3f} (oracle-tonic frozen result: 0.951)")
print(f"  M1 estimated rec_top1: {interp(m1_sweep, REAL_TONIC_ERROR_CENTS):.3f} (oracle-tonic frozen result: 0.984)")

(ROOT / "results").mkdir(exist_ok=True)
summary = {
    "real_tonic_error_cents_median": REAL_TONIC_ERROR_CENTS,
    "b1_sweep": b1_sweep.to_dict(orient="records"),
    "m1_sweep": m1_sweep.to_dict(orient="records"),
}
(ROOT / "results" / "phase4a_tonic_robustness.json").write_text(json.dumps(summary, indent=2, default=float))
print("saved results/phase4a_tonic_robustness.json")